# NurseGemma Quick Test

Run this on **Kaggle** or **Google Colab** with GPU enabled.

**Setup:**
- Kaggle: Settings → GPU T4, Internet On, add HF_TOKEN secret
- Colab: Runtime → GPU, run `login()` cell

In [ ]:
# Install dependencies
!pip install -q transformers accelerate requests pillow

In [ ]:
# Setup
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import torch
import time
import requests
from PIL import Image
from io import BytesIO
from transformers import AutoProcessor, AutoModelForImageTextToText
from huggingface_hub import login

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Authenticate - CHOOSE ONE:

# Option A: Kaggle (uses secret)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    login(token=secrets.get_secret('HF_TOKEN'))
    print('Authenticated via Kaggle secrets')
except:
    # Option B: Colab/Local - interactive login
    login()
    print('Authenticated interactively')

In [ ]:
# Load MedGemma 1.5 4B
MODEL_ID = 'google/medgemma-1.5-4b-it'

print('Loading MedGemma...')
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True
)
print(f'Ready on {next(model.parameters()).device}')

In [ ]:
# Helper functions
def ask(prompt, max_tokens=500):
    messages = [{'role': 'user', 'content': [{'type': 'text', 'text': prompt}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, 
                                           tokenize=True, return_dict=True, return_tensors='pt'
                                          ).to(model.device, dtype=torch.bfloat16)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    return processor.decode(out[0], skip_special_tokens=True)

def analyze_img(url, prompt, max_tokens=800):
    img = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert('RGB')
    messages = [{'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': prompt}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True,
                                           tokenize=True, return_dict=True, return_tensors='pt'
                                          ).to(model.device, dtype=torch.bfloat16)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    return processor.decode(out[0], skip_special_tokens=True)

print('Helper functions ready')

In [ ]:
# TEST 1: Explain CHF to family
print('TEST 1: Explain CHF to family')
print('=' * 50)

start = time.time()
result = ask('''Explain CHF (Congestive Heart Failure) to a worried family member.
Use simple terms, 8th grade reading level, helpful analogies.''')
print(f'Time: {time.time()-start:.1f}s\n')
print(result)

In [ ]:
# TEST 2: Critical Lab Analysis
print('TEST 2: Critical Lab (Digoxin + Low K+)')
print('=' * 50)

start = time.time()
result = ask('''Patient: 68F with CHF on Digoxin 0.125mg daily
Labs: K+ 3.2 (low), Digoxin level 1.8 (high therapeutic)
Vitals: BP 92/58, HR 52

1. What's the clinical concern?
2. Priority: Routine/Urgent/Critical?
3. What should the nurse do?
4. SBAR for MD notification''')
print(f'Time: {time.time()-start:.1f}s\n')
print(result)

In [ ]:
# TEST 3: CXR Analysis (MULTIMODAL)
print('TEST 3: Chest X-Ray Analysis')
print('=' * 50)

CXR = 'https://upload.wikimedia.org/wikipedia/commons/8/87/X-ray_of_lobar_pneumonia.jpg'
print(f'Image: {CXR}\n')

start = time.time()
result = analyze_img(CXR, '''Analyze this chest X-ray:
1. Key findings
2. Clinical significance
3. Nursing implications
4. Urgency: ROUTINE/ATTENTION/URGENT/CRITICAL''')
print(f'Time: {time.time()-start:.1f}s\n')
print(result)

In [ ]:
# TEST 4: Wound Assessment (MULTIMODAL)
print('TEST 4: Wound Assessment')
print('=' * 50)

WOUND = 'https://upload.wikimedia.org/wikipedia/commons/f/fc/Grade3.jpg'
print(f'Image: {WOUND}\n')

start = time.time()
result = analyze_img(WOUND, '''Analyze this pressure injury per NPIAP guidelines:
1. Staging (1/2/3/4/Unstageable/DTPI) with justification
2. Tissue types visible
3. Wound characteristics
4. Nursing recommendations
5. Urgency level''')
print(f'Time: {time.time()-start:.1f}s\n')
print(result)

In [ ]:
# Summary
print('\n' + '=' * 50)
print('NURSEGEMMA TEST COMPLETE')
print('=' * 50)
print('''
✓ Text inference working
✓ Multimodal (image) inference working
✓ MedGemma 1.5 4B loaded and functional

Ready for competition submission!
''')